# Explainable AML Detection — GPU run on Colab

Runs stages 2-3 (training + per-typology evaluation) and stages 4-5 (GNNExplainer
+ fidelity) of the CSF 3127 project on the **full 5,078,345-edge HI-Small graph**.

Stage 1 (graph construction) already ran to completion locally, so the raw 475 MB
`HI-Small_Trans.csv` is **not** needed here — only ~191 MB of prebuilt tensors.

**Runtime → Change runtime type → T4 GPU** before running anything.

### What you need in Google Drive first

Create a folder `MyDrive/gnn-aml/` containing:

| file | source | size |
|---|---|---|
| `gnn_aml_code.zip` | `colab/gnn_aml_code.zip` in the repo | 48 KB |
| `hismall_graph.pt` | `data/processed/` | 151 MB |
| `hismall_splits.pkl` | `data/processed/` | 40 MB |
| `hismall_account_maps.pkl` | `data/processed/` | 13 MB *(optional — nothing downstream reads it)* |


## 1. Confirm the GPU


In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'gpu: {p.name}, {p.total_memory / 2**30:.1f} GB')
else:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')


## 2. Mount Drive and unpack the code


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, zipfile, shutil, pathlib

SRC = '/content/drive/MyDrive/gnn-aml'
DST = '/content/gnn-aml-detection'

os.makedirs(DST, exist_ok=True)
with zipfile.ZipFile(f'{SRC}/gnn_aml_code.zip') as z:
    z.extractall(DST)
os.chdir(DST)

pathlib.Path('data/processed').mkdir(parents=True, exist_ok=True)
REQUIRED = ['hismall_graph.pt', 'hismall_splits.pkl']
OPTIONAL = ['hismall_account_maps.pkl']  # not read by train.py or explain.py

for f in REQUIRED:
    if not os.path.exists(f'{SRC}/{f}'):
        raise FileNotFoundError(f'{SRC}/{f} is missing — upload it to Drive first.')
    if not os.path.exists(f'data/processed/{f}'):
        shutil.copy(f'{SRC}/{f}', f'data/processed/{f}')
        print('copied', f)

for f in OPTIONAL:
    if os.path.exists(f'{SRC}/{f}') and not os.path.exists(f'data/processed/{f}'):
        shutil.copy(f'{SRC}/{f}', f'data/processed/{f}')
        print('copied', f)

for d in ['outputs/checkpoints', 'outputs/metrics', 'outputs/explanations', 'outputs/reports']:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

print(os.getcwd())
print(sorted(os.listdir('data/processed')))


## 3. Install PyTorch Geometric

Colab ships torch already; only PyG is missing. Do **not** reinstall torch — that
swaps the CUDA build out from under the runtime and forces a restart.


In [ ]:
!pip install -q torch_geometric matplotlib networkx
import torch_geometric, matplotlib, networkx
print('pyg:', torch_geometric.__version__)
print('matplotlib:', matplotlib.__version__, '| networkx:', networkx.__version__)


## 4. Sanity check: the fixture test suite

16 tests against the synthetic fixture. Fast, and it catches a broken unpack or a
PyG version mismatch before you spend GPU time on it. Four of them pin the
explainer's L-hop subgraph extraction, including that subgraph logits match
full-graph logits — the equivalence the memory fix depends on.


In [ ]:
!PYTHONPATH=. python -m pytest tests/ -q


## 5. Train both architectures on the full graph

`HeterophilyGNN` and `BaselineSAGE`, three seeds each, on all 5,078,345 edges.

The baseline is SRS **FR-5**, and without it the §174 success criterion
("measurably better recall/F1 than the homophily baseline") cannot be
evaluated at all — nor can §211's "no improvement is a valid negative result",
which is explicitly conditional on the baseline being included.

The two configs are identical apart from `model.type`; `tests/test_configs.py`
asserts that, so a stray hyperparameter drift can't quietly invalidate the
comparison.


In [ ]:
import os, subprocess, sys


def run(cmd, log_path):
    """Stream a subprocess to the cell and to a log file, raising on failure.

    Uses an argument list rather than `!cmd $VAR`: IPython's $VAR interpolation
    does not reliably survive a for-loop body and substitutes an empty string
    when it fails, which produces an argparse error and no output at all.
    """
    env = dict(os.environ, PYTHONPATH='.')
    with open(log_path, 'w') as fh:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, env=env)
        for line in proc.stdout:
            print(line, end='')
            fh.write(line)
        proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f'FAILED (exit {proc.returncode}); see {log_path}')


CONFIGS = {
    'heterophily': 'configs/heterophily_gnn_hismall.yaml',
    'baseline':    'configs/baseline_sage_hismall.yaml',
}
SEEDS = (42, 43, 44)

# Six runs: two architectures x three seeds. Three seeds because CUDA's scatter
# reductions are non-deterministic, so a single run of each would compare two
# points drawn from distributions that overlap — earlier heterophily runs
# spanned AUPRC 0.026-0.040 with the config unchanged.
for name, cfg in CONFIGS.items():
    for seed in SEEDS:
        print(f'\n{"=" * 24} {name} seed {seed} {"=" * 24}', flush=True)
        run([sys.executable, '-u', 'scripts/train.py',
             '--config', cfg, '--seed', str(seed)],
            f'outputs/reports/train_{name}_seed{seed}.log')

print('\nAll six runs finished.')


### Compare the two architectures

Pools `outputs/metrics/*_overall.csv` — one row per run. Judge on **AUPRC**
against the 0.00177 test base rate, and on illicit-class recall/F1 per SRS
§174. Accuracy is shown beside `trivial_accuracy` (the constant "always
licit" predictor) purely to make clear why it is not the metric here.


In [ ]:
import glob
import numpy as np
import pandas as pd

rows = [pd.read_csv(f) for f in glob.glob('outputs/metrics/*_overall.csv')]
if not rows:
    raise SystemExit('No *_overall.csv — the training cell above did not finish.')
runs = pd.concat(rows, ignore_index=True)
BASE_RATE = 1797 / 1015669
runs['auprc_lift'] = runs['auprc'] / BASE_RATE

print('per-run results')
display(runs[['model', 'seed', 'auprc', 'auprc_lift', 'recall',
              'precision', 'f1', 'accuracy', 'trivial_accuracy']]
        .sort_values(['model', 'seed']).round(5))

agg = runs.groupby('model').agg(
    auprc_mean=('auprc', 'mean'), auprc_std=('auprc', 'std'),
    auprc_min=('auprc', 'min'), auprc_max=('auprc', 'max'),
    recall_mean=('recall', 'mean'), f1_mean=('f1', 'mean'), n=('auprc', 'size'),
)
agg['auprc_lift'] = agg['auprc_mean'] / BASE_RATE
print('\npooled across seeds')
display(agg.round(5))

# Does the difference clear the seed-to-seed noise?
if set(agg.index) >= {'heterophily_gnn', 'baseline_sage'}:
    h = runs[runs.model == 'heterophily_gnn']['auprc']
    b = runs[runs.model == 'baseline_sage']['auprc']
    diff = h.mean() - b.mean()
    spread = max(h.max() - h.min(), b.max() - b.min())
    print(f'\nheterophily - baseline AUPRC: {diff:+.5f}')
    print(f'largest within-model seed spread: {spread:.5f}')
    verdict = 'LARGER than' if abs(diff) > spread else 'WITHIN'
    print(f'the difference is {verdict} seed noise', end=' ')
    print('-> a real effect' if abs(diff) > spread else
          '-> not separable at n=3; report as no measurable difference')
    print('\nn=3 per arm is a small sample. Treat this as a sanity check on',
          'effect size, not a significance test.')


### Compare the two architectures

Pools the six runs and reports mean ± std per architecture. Read the spread
before the difference: with three seeds, a gap smaller than the seed-to-seed
std is not a result.


In [ ]:
import glob
import numpy as np
import pandas as pd
from scipy import stats

N_TEST, N_ILLICIT = 1015669, 1797

rows = []
for log in sorted(glob.glob('outputs/reports/train_*_seed*.log')):
    name = log.split('train_')[1].rsplit('_seed', 1)[0]
    seed = int(log.rsplit('seed', 1)[1].split('.')[0])
    text = open(log).read()
    if '--- Overall test metrics ---' not in text:
        print(f'skipping {log}: no test metrics (did it fail?)')
        continue
    blob = text.split('--- Overall test metrics ---')[1].split('\n')[1]
    d = eval(blob, {'np': np})  # the script prints a plain dict repr
    rows.append({'model': name, 'seed': seed, **{k: float(v) for k, v in d.items()}})

res = pd.DataFrame(rows)
res['auprc_lift'] = res['auprc'] / (N_ILLICIT / N_TEST)
print(res.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

print('\n--- mean +/- std over seeds ---')
agg = res.groupby('model')[['precision', 'recall', 'f1', 'auprc', 'auprc_lift']].agg(['mean', 'std'])
display(agg.round(4))

# SRS §174 asks whether the heterophily model is *measurably* better on the
# illicit class. With 3 seeds a t-test is weak; report the effect alongside
# the spread and let the reader judge.
if set(res['model']) >= {'heterophily', 'baseline'}:
    for metric in ('recall', 'f1', 'auprc'):
        h = res[res.model == 'heterophily'][metric]
        b = res[res.model == 'baseline'][metric]
        t = stats.ttest_ind(h, b, equal_var=False)
        print(f'{metric:>9}: heterophily {h.mean():.4f} +/- {h.std():.4f} | '
              f'baseline {b.mean():.4f} +/- {b.std():.4f} | '
              f'diff {h.mean() - b.mean():+.4f} | p={t.pvalue:.3f}')
    print('\nn=3 per arm. Treat p-values as descriptive, not confirmatory.')


## 6. Explanations + per-typology fidelity

Point `--checkpoint` at whatever stage 5 wrote into `outputs/checkpoints/`.

This runs three times under seeds 42/43/44. Each writes its own
`fidelity_..._seed{N}.csv`, so nothing overwrites anything, and the next cell
pools them into a mean with a confidence interval.


In [ ]:
import glob, os, subprocess, sys

# Checkpoint discovery lives in THIS cell on purpose. Reading it from an
# earlier cell means a kernel restart silently leaves it stale or unset.
# There are now six checkpoints. Select by the model type each one records,
# rather than by filename or by taking the last glob hit.
import torch

EXPLAIN_MODEL = 'heterophily_gnn'
ckpts = []
for path in sorted(glob.glob('outputs/checkpoints/*.pt')):
    meta = torch.load(path, weights_only=False, map_location='cpu')
    if meta.get('config', {}).get('model', {}).get('type') == EXPLAIN_MODEL:
        ckpts.append(path)
if not ckpts:
    raise SystemExit(f'No {EXPLAIN_MODEL} checkpoint found — run the training cell first.')
CKPT = ckpts[0]
print(f'{len(ckpts)} {EXPLAIN_MODEL} checkpoints; explaining {CKPT}')

# GNNExplainer's mask optimisation is stochastic. One run gives a point
# estimate with no error bar — the first two runs of this pipeline disagreed
# by up to 0.35 mean fidelity. Three seeds against the SAME checkpoint
# isolates explainer variance from training variance.
#
# Invoked via subprocess with an argument list rather than `!... $CKPT`:
# IPython's $VAR interpolation does not reliably survive a for-loop body,
# and when it fails it substitutes an empty string, so argparse errors out
# and the run produces no CSV at all.
env = dict(os.environ, PYTHONPATH='.')

for seed in (42, 43, 44):
    print(f'\n{"=" * 30} seed {seed} {"=" * 30}', flush=True)
    log_path = f'outputs/reports/explain_seed{seed}.log'
    cmd = [
        sys.executable, '-u', 'scripts/explain.py',
        '--config', 'configs/heterophily_gnn_hismall.yaml',
        '--checkpoint', CKPT,
        '--seed', str(seed),
    ]
    with open(log_path, 'w') as fh:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, env=env)
        for line in proc.stdout:
            print(line, end='')
            fh.write(line)
        proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f'seed {seed} failed (exit {proc.returncode}); see {log_path}')

print('\nAll three seeds finished.')
print(sorted(glob.glob('outputs/explanations/fidelity_*_seed*.csv')))


### Look at the explanations

Pools the three seeds into mean fidelity with a 95% CI, and shows the per-seed
means side by side. **Read `seed_spread` before quoting any mean** — if it is
as large as the gaps between typologies, the ranking is noise at this sample
size and belongs in the writeup as a limitation, not a result.


In [ ]:
import glob, os
import numpy as np
import pandas as pd
from IPython.display import Image, display

paths = sorted(glob.glob('outputs/explanations/fidelity_*_seed*.csv'))
if not paths:
    raise SystemExit(
        'No fidelity_*_seed*.csv found. The explain cell above did not produce '
        'output — scroll up and read its error rather than this one.'
    )
runs = []
for path in paths:
    d = pd.read_csv(path)
    d['seed'] = int(path.rsplit('seed', 1)[1].split('.')[0])
    runs.append(d)
df = pd.concat(runs, ignore_index=True)
print(f'{len(paths)} seeds, {len(df)} explanations total')

# Per-seed means first, then pool: shows whether seeds agree at all.
per_seed = df.dropna(subset=['fidelity']).groupby(['typology', 'seed'])['fidelity'].mean().unstack()
display(per_seed.style.format('{:.3f}').set_caption('mean fidelity per seed'))

g = df.dropna(subset=['fidelity']).groupby('typology')['fidelity']
summary = pd.DataFrame({'mean': g.mean(), 'std': g.std(), 'n': g.count()})
summary['sem'] = summary['std'] / np.sqrt(summary['n'])
summary['ci95_lo'] = summary['mean'] - 1.96 * summary['sem']
summary['ci95_hi'] = summary['mean'] + 1.96 * summary['sem']
summary['seed_spread'] = per_seed.max(axis=1) - per_seed.min(axis=1)
display(summary[['mean', 'ci95_lo', 'ci95_hi', 'n', 'seed_spread']].round(3))
print('\nseed_spread is max-minus-min of the per-seed means. If it is comparable')
print('to the gaps between typologies, the ranking is not measurable at this n.')

for chart in sorted(glob.glob('outputs/reports/figures/*_fidelity_by_typology.png')):
    display(Image(chart))

# One representative explanation per typology, from the first seed.
first = df[df['seed'] == df['seed'].min()].dropna(subset=['typology', 'fidelity'])
for typ, grp in first.groupby('typology'):
    best = grp.loc[grp['fidelity'].idxmax()]
    img = best.get('subgraph_image')
    if isinstance(img, str) and img and os.path.exists(img):
        print(f"{typ} — best fidelity {best['fidelity']:.3f} (edge {int(best['edge_idx'])})")
        display(Image(img))


## 7. Copy results back to Drive

Colab runtimes are wiped on disconnect. Run this before you close the tab.


In [ ]:
import shutil, os
OUT = '/content/drive/MyDrive/gnn-aml/outputs'
if os.path.exists(OUT):
    shutil.rmtree(OUT)
shutil.copytree('outputs', OUT)
print('saved to', OUT)
for root, _, files in os.walk(OUT):
    for f in files:
        print(' ', os.path.join(root, f).replace(OUT, ''))


## Fallback: if the full graph OOMs on the GPU

Peak activation for the edge head at 5.08M edges is roughly 2.6 GB for the
concatenated edge representation plus ~1.3 GB per hidden layer, and the two
directional aggregators each materialise a [E, 64] message tensor. That should fit
a T4's 15 GB, but it isn't guaranteed.

Two options, in order of preference:

1. **Halve the model width.** `hidden_dim: 32` in the config roughly halves every
   edge-side activation and keeps all 5.08M edges — you lose model capacity, not data.
2. **Fall back to the subsample.** Upload `hismall_sub2000000_*` from
   `data/processed/` and use `configs/heterophily_gnn_hismall_sub2m.yaml`. Note that
   config takes the *earliest* 2M transactions, which hold only 984 of the 5,177
   illicit edges — see the note in that file.

**The explain stage no longer needs full-graph memory.** It extracts each
target edge's L-hop computation subgraph (a few thousand edges) instead of
masking all 5.08M, so if `explain.py` OOMs, something else is wrong — don't
reach for the subsample to fix it.


In [ ]:
# Option 1 — retry at half width, still on the full graph.
# !sed -i 's/hidden_dim: 64/hidden_dim: 32/' configs/heterophily_gnn_hismall.yaml
# !PYTHONPATH=. python -u scripts/train.py --config configs/heterophily_gnn_hismall.yaml

# Option 2 — subsampled graph (upload hismall_sub2000000_* to Drive first).
# !PYTHONPATH=. python -u scripts/train.py --config configs/heterophily_gnn_hismall_sub2m.yaml
